<a href="https://colab.research.google.com/github/Gr1lledChee5e/LLM/blob/main/Model_Fine_Tuning_With_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAS 691 - Large Language Models
## Day 8 - Model Fine-Tuning with OpenAI

+ April 8, 2026

In this script, we will fine-tune a model through OpenAI. Reference: https://developers.openai.com/api/docs/guides/supervised-fine-tuning

1. Prepare the training data
2. Upload training data to OpenAI
3. Fine-Tune
4. Use fine-tuned model
5. Evaluate our fine-tuned model (simple)




## Setup and Goals

**BrightWave Mobile needs your help!**

In this example we fine-tune a language model for BrightWave Mobile, a telecom company that wants to automatically process customer complaints from channels like Twitter or Facebook. Given a customer complaint, the model should:

1. Classify the issue type
2. Assign a severity level (low, medium, high)
3. Generate a professional, empathetic response that acknowledges the issue, maintains a calm and positive tone, and guides the customer toward next steps (e.g., contacting support via DM).

The goal of this exercise is to demonstrate how supervised fine-tuning can improve consistency, structure, and tone control in real-world applications, while reinforcing key concepts such as prompt design, labeled data construction, and evaluation of model outputs against business and communication objectives.



## Background - `completions` API

We have been using the `responses` API. It is newer, with cleaner inputs and outputs.

For supervised fine-tuning, however, the training data must be formatted as a JSONL file where each line contains a messages array in the form historically used by the OpenAI `chat.completions` API (precursor to `responses`). This API made use of `system` (instructions for the model), `user` (user prompt), and `assistant` (typically prior output or chat history) roles.

Even though you will use the newer OpenAI API at inference time, the fine-tuning process still relies on this chat-style message format because it explicitly teaches the model the mapping from prior messages (*system* + *user*) to the desired *assistant* response, just as the `chat.completions` API was designed to do.

In [ ]:
from openai import OpenAI
from google.colab import userdata
api_key = userdata.get('gpt4m')
client = OpenAI(api_key = api_key)

In [ ]:
instructions = "You are a telecom customer support assistant. Classify the issue type, assign a severity (low, medium, high), and draft a response that is empathetic, calm, and helpful."
complaint = "My internet is down. You guys are the worst!"

# Call with responses
reposnse = client.chat.completions.create(
    model = "gpt-4o-mini",
    instructions = instructions,
    input = complaint
)

# Call with completions
completion = client.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": complaint}
    ]
)
# Call with completions




TypeError: Missing required arguments; Expected either ('messages' and 'model') or ('messages', 'model' and 'stream') arguments to be given

In [ ]:
print(response.output_text)

NameError: name 'response' is not defined

In [ ]:
print(result.choices[0].message.content)

NameError: name 'result' is not defined

## 1. Prepare the Training Data

Each approach to fine-tuning will require your data in a specific format. For OpenAI you need to prepare a `JSONL` file. This is basically a file with one dictionary per row, where each dictionary is the `messages` portion you are used to inputting in the `client.chat.completions` api. It is used as "labeled data" so the model can learn - based on these instructions (system role) and this complaint (user role), this is the desired ouptut (assistant role).

Each dictionary serves as one training sample. You are basically passing it a set of prompts and responses so it can learn what responses should look like.

The `complaints_data` below represents true, historical data. This will allow the model to see complaints, how they were classified, and the response, so it can learn that same pattern.


In [ ]:
import pandas as pd
import json

In [ ]:
complaints_data = pd.read_csv('https://dxl-datasets.s3.us-east-1.amazonaws.com/masllm/telecom_fine_tuning.csv')
complaints_data.head()

,record_id,company,channel,customer_message,issue_type,severity,needs_private_followup,response_draft
0,TW001,BrightWave Mobile,X,My home internet has been down since yesterday...,outage,high,yes,We're very sorry your home internet has been d...
1,TW002,BrightWave Mobile,Facebook,Been on hold for 45 minutes just to ask why my...,billing_issue,medium,yes,We're sorry for the long hold time and the sur...
2,TW003,BrightWave Mobile,X,Unlimited data but somehow videos won't load u...,slow_data,medium,yes,We're sorry your data service isn't performing...
3,TW004,BrightWave Mobile,Google Reviews,The store rep promised my activation would be ...,installation_delay,high,yes,We're very sorry your activation took far long...
4,TW005,BrightWave Mobile,X,Roaming in Canada and my phone hasn't worked s...,roaming_issue,high,yes,We're sorry your service isn't working while y...


In [ ]:
# I used the system message below when generating the "true" data that we use to train, so I'll stick with it. You could imagine other personalities you could give a customer support assistant. Could even tailor the language to geographic specifics if warranted.
complaints_data.response_draft[0]

"We're very sorry your home internet has been down, especially while you're trying to work from home. We know how disruptive that is. Please send us a DM with your service address and the best contact number, and we'll check the outage status and account notes right away. We appreciate your patience and want to get you back online as quickly as possible."

In [ ]:
system_message = "You are a telecom customer support assistant. Classify the issue type, assign a severity (low, medium, high), and draft a response that is empathetic, calm, and helpful."

def build_training_rows(row):
  user_message = row['customer_message']

  issue_type = row['issue_type']
  severity = row['severity']
  response_draft = row['response_draft'] # Corrected: Removed raise_response and used correct column name

  assistant_message = f"Issue Type: {issue_type}\nSeverity: {severity}\nResponse Draft: {response_draft}"

  return {
      "messages":[{"role":"system", "content":system_message}, {"role":"user", "content":user_message}, {"role":"assistant", "content":assistant_message}]
  }

In [ ]:
telecom_training_data = complaints_data.apply(build_training_rows, axis=1)
telecom_training_data.iloc[0]

{'messages': [{'role': 'system',
   'content': 'You are a telecom customer support assistant. Classify the issue type, assign a severity (low, medium, high), and draft a response that is empathetic, calm, and helpful.'},
  {'role': 'user',
   'content': 'My home internet has been down since yesterday and I work from home. Your outage map says everything is fine. What is going on?'},
  {'role': 'assistant',
   'content': "Issue Type: outage\nSeverity: high\nResponse Draft: We're very sorry your home internet has been down, especially while you're trying to work from home. We know how disruptive that is. Please send us a DM with your service address and the best contact number, and we'll check the outage status and account notes right away. We appreciate your patience and want to get you back online as quickly as possible."}]}

In [ ]:
# Save telecom_training_data as a jsonl file for use with openai
# Find file and double click on it to view the structure
with open('telecom_training_data.jsonl', 'w') as file:
    for item in telecom_training_data:
        json_line = json.dumps(item)
        file.write(json_line + '\n')

## 2. Upload Training Data to OpenAI

You can upload this in the developer portal or through the API using `files.create()`

Reference: https://developers.openai.com/api/reference/python/resources/files/methods/create

In [ ]:
client.files.create(
  file=open("telecom_training_data.jsonl", "rb"),
  purpose="fine-tune",
  expires_after={
    "anchor": "created_at",
    "seconds": 2592000
  }
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ft:gpt-4********************************Qedq. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}, 'detail': {'message': 'Incorrect API key provided: ft:gpt-4********************************Qedq. You can find your API key at https://platform.openai.com/account/api-keys.', 'code': 'invalid_api_key'}}

In [ ]:
# files.create() will send your jsonl file to your storage space in the openai platform
#  returns a file id that you will use next
client.files.create(
  file=open("telecom_training_data.jsonl", "rb"),
  purpose="fine-tune"
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ft:gpt-4********************************Qedq. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}, 'detail': {'message': 'Incorrect API key provided: ft:gpt-4********************************Qedq. You can find your API key at https://platform.openai.com/account/api-keys.', 'code': 'invalid_api_key'}}

## 3. Fine-Tune

Once the training data is loaded, we can begin fine-tuning. You can track the status in the fine-tuning sections of your dashboard on OpenAI (you can also track via api calls from python). In my own experience, several models crashed because I had the data in the wrong format, but once that was sorted out it all went pretty smooth/quick. I've also recently been seeing more jobs crash because they violate OpenAI policies.

**What model to fine-tune?**

I am going to fine-tune `gpt-4o-mini`, for a balance of cost and performance.

OpenAI does not allow you to fine-tune all of their models. You can compare models here - https://developers.openai.com/api/docs/models/compare - to see if v1/fine-tuning is selected under `endpoints`. As of now, the GPT-5 models are not.

You can also fine-tune your fine-tuned models if you get more data later and want to continue training.

In [ ]:
# Create a fine tuning job
client.fine_tuning.jobs.create(
  training_file='file-GJ3arpbS5Ubrpr48Rbk9kt',
  model="gpt-4o-mini-2024-07-18"
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ft:gpt-4********************************Qedq. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## 4. Use your fine-tuned model

Once your fine-tuning job is complete, you can find it in the Fine-Tuning sections of the Dashboard. Your model will have an associated "Output model" that you can now use when making calls. Here, I am including the model ID from a previous fine-tuning job that I ran on the data above so you can use it without waiting for the above to complete...

In [ ]:
our_fine_tuned_model = 'ft:gpt-4o-mini-2024-07-18:personal::DS51Qedq'

In [ ]:
response = client.responses.create(
    model = our_fine_tuned_model,
    instructions = "You are a telecom customer support assistant. Classify the issue type, assign a severity (low, medium, high), and draft a response that is empathetic, calm, and helpful.",
    input = "MAKE UP A CUSTOMR COMPLAINT"
)

print(response.output_text)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `ft:gpt-4o-mini-2024-07-18:personal::DS51Qedq` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

## 5. Evaluate Model

We do not have a full validation set to use here, but let's try with one question/answer set from the bottom of the interview that was not included in the training data. In practice, you would have several Q&A pairs.

The `evaluate` library is maintained by HuggingFace and has most metrics needed. It was designed to make evaluating language models easier and more standardized.

In [ ]:
# Needed
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=350aae041bb717a6e6cf845156e9503e2f785476cb270053ff858decfb6d3f34
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# I think not needed...
#!pip install -U evaluate datasets huggingface_hub

In [ ]:
import evaluate
import nltk

# Download required NLTK data for METEOR
nltk.download('punkt')
nltk.download('wordnet')

# Load the METEOR metric
meteor = evaluate.load('meteor')
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')

In [ ]:
complaints_data.head()

,record_id,company,channel,customer_message,issue_type,severity,needs_private_followup,response_draft
0,TW001,BrightWave Mobile,X,My home internet has been down since yesterday...,outage,high,yes,We're very sorry your home internet has been d...
1,TW002,BrightWave Mobile,Facebook,Been on hold for 45 minutes just to ask why my...,billing_issue,medium,yes,We're sorry for the long hold time and the sur...
2,TW003,BrightWave Mobile,X,Unlimited data but somehow videos won't load u...,slow_data,medium,yes,We're sorry your data service isn't performing...
3,TW004,BrightWave Mobile,Google Reviews,The store rep promised my activation would be ...,installation_delay,high,yes,We're very sorry your activation took far long...
4,TW005,BrightWave Mobile,X,Roaming in Canada and my phone hasn't worked s...,roaming_issue,high,yes,We're sorry your service isn't working while y...


In [ ]:
# Taking one sample to evaluate (in practice, you would have a separate holdout set for evaluations, you'd validate on an entire holdout dataset, not just a single test observation as I'm doing here)
telecom_training_data.iloc[0]

{'messages': [{'role': 'system',
   'content': 'You are a telecom customer support assistant. Classify the issue type, assign a severity (low, medium, high), and draft a response that is empathetic, calm, and helpful.'},
  {'role': 'user',
   'content': 'My home internet has been down since yesterday and I work from home. Your outage map says everything is fine. What is going on?'},
  {'role': 'assistant',
   'content': "Issue Type: outage\nSeverity: high\nResponse Draft: We're very sorry your home internet has been down, especially while you're trying to work from home. We know how disruptive that is. Please send us a DM with your service address and the best contact number, and we'll check the outage status and account notes right away. We appreciate your patience and want to get you back online as quickly as possible."}]}

In [ ]:
complaint = '''My home internet has been down since yesterday and I work from home. Your outage map says everything is fine. What is going on?'''
response = '''
  Issue Type: outage
  Severity: high
  Response Draft: We're very sorry your home internet has been down, especially while you're trying to work from home. We know how disruptive that is. Please send us a DM with your service address and the best contact number, and we'll check the outage status and account notes right away. We appreciate your patience and want to get you back online as quickly as possible.
'''

In [ ]:
response_custom_model = client.responses.create(
    model = our_fine_tuned_model,
    instructions = system_message,
    input = complaint
)

print(response_custom_model.output_text)

Issue Type: service_interruption
Severity: high
Response Draft: We're very sorry your home internet has been down and that the outage map hasn't reflected the issue. We know how disruptive that is when you're working from home. Please DM us your service address and a contact number so we can check the status and troubleshoot with you. We appreciate your patience and want to get you back online as quickly as possible.


In [ ]:
response_standard_model = client.responses.create(
    model = 'gpt-4o-mini',
    instructions = system_message,
    input = complaint
)

print(response_standard_model.output_text)

**Issue Type:** Internet Service Disruption  
**Severity:** High  

---

**Response:**

Hi there,

I completely understand how frustrating it can be to experience internet service disruptions, especially when you're working from home. Thank you for bringing this to our attention.

While our outage map indicates that services are functioning normally, there can sometimes be localized issues that aren’t reflected on the map right away. I recommend a few troubleshooting steps that might help resolve the problem:

1. **Restart Your Modem/Router:** Unplug it from the power source, wait for about 30 seconds, and plug it back in. This can often resolve connectivity issues.
   
2. **Check Connections:** Ensure all cables are securely connected and there are no visible damages.

3. **Device Check:** If possible, see if other devices in your home can connect to the internet. This helps determine if the issue is with the network or a specific device.

If the issue persists after these steps, plea

### ROUGE SCORE

A ROUGE score (Recall-Oriented Understudy for Gisting Evaluation) is a set of metrics used to **evaluate automatically generated text** (like summaries or answers) by comparing it to reference texts, **focusing on overlap of content**.

ROUGE is widely used in tasks like summarization and text generation to quantify similarity between generated and “ground truth” outputs, often alongside other metrics or human evaluation.

In [ ]:
rouge_custom_model = rouge.compute(predictions = [response_custom_model.output_text], references = [response])
rouge_standard_model = rouge.compute(predictions = [response_standard_model.output_text], references = [response])

In [ ]:
print(rouge_custom_model)
print(rouge_standard_model)

{'rouge1': np.float64(0.8108108108108107), 'rouge2': np.float64(0.5753424657534246), 'rougeL': np.float64(0.6756756756756757), 'rougeLsum': np.float64(0.6756756756756757)}
{'rouge1': np.float64(0.3450704225352113), 'rouge2': np.float64(0.07801418439716312), 'rougeL': np.float64(0.16901408450704222), 'rougeLsum': np.float64(0.2676056338028169)}


### BLEU SCORE

The BLEU score (Bilingual Evaluation Understudy) is a metric for evaluating generated text by comparing it to one or more reference texts, based on n-gram precision (e.g., word and phrase overlap). It includes a brevity penalty to discourage short outputs and typically ranges from 0 to 1, where higher scores indicate closer matches to references. It's widely used in machine translation, but also applies to summarization and other text generation tasks.

In [ ]:
bleu_custom_model = bleu.compute(predictions = [response_custom_model.output_text], references = [response])
bleu_standard_model = bleu.compute(predictions = [response_standard_model.output_text], references = [response])

In [ ]:
print(bleu_custom_model)
print(bleu_standard_model)

{'bleu': 0.5519764586035478, 'precisions': [0.8, 0.6075949367088608, 0.47435897435897434, 0.4025974025974026], 'brevity_penalty': 1.0, 'length_ratio': 1.0126582278481013, 'translation_length': 80, 'reference_length': 79}
{'bleu': 0.0370474471773131, 'precisions': [0.1947565543071161, 0.05639097744360902, 0.022641509433962263, 0.007575757575757576], 'brevity_penalty': 1.0, 'length_ratio': 3.3797468354430378, 'translation_length': 267, 'reference_length': 79}


### METEOR SCORE

METEOR (Metric for Evaluation of Translation with Explicit ORdering) is a text evaluation metric designed to improve on BLEU by **better aligning with human judgment**. Like ROUGE and BLEU, it compares a generated text to a reference, but it goes further by allowing flexible matches, including stemming (e.g., “run” vs. “running”), synonyms, and paraphrases.

In practice, METEOR is often used in machine translation and text generation because it:

+ captures semantic similarity better than BLEU,
+ balances precision and recall (unlike BLEU or ROUGE alone),
+ and correlates more closely with human evaluations.

In [ ]:
meteor_custom_model = meteor.compute(predictions = [response_custom_model.output_text], references = [response])
meteor_standard_model = meteor.compute(predictions = [response_standard_model.output_text], references = [response])

In [ ]:
print(f"Custom Model: {meteor_custom_model['meteor']}\nStandard Model: {meteor_standard_model['meteor']}")

Custom Model: 0.8120291857316078
Standard Model: 0.316903312274986


## Last years model / just for fun

In [ ]:
response = client.responses.create(
    model = 'ft:gpt-4o-mini-2024-07-18:personal::BJtsW2OZ',
    instructions = 'You are a politician being interviewed by a reporter.',
    input = 'What is your favorite color?'
)
print(response.output_text)

I think purple is a beautiful color. It has a softness to it, but it’s very powerful. And it’s the color of the future, I will say that. First they said, “Gee, you know, it’s a little purple in there.” People came back to me, they said, “That was the greatest color scheme that we’ve ever seen in a building.”


In [ ]:
response = client.responses.create(
    model = 'ft:gpt-4o-mini-2024-07-18:personal::BJtsW2OZ',
    instructions = 'You are a politician being interviewed by a reporter.',
    input = 'Who is your favorite golfer?'
)
print(response.output_text)

I have a lot of them. I think, you know, I’d probably have to say Tiger, but I also think Rory has been really great. Phil Mickelson, I really like Phil. You know, I think probably Phil and Tiger are tied for first, and then probably Rory is. But I’ve always liked Tiger, I’ve always liked Phil, and I’ve always liked Rory.


In [ ]:
response = client.responses.create(
    model = 'ft:gpt-4o-mini-2024-07-18:personal::BJtsW2OZ',
    instructions = 'You are a politician being interviewed by a reporter.',
    input = 'How do you feel about vegetables?'
)
print(response.output_text)

I like a lot of vegetables, actually. In fact, I like carrots. I mean, look at me. I have orange all over. I could have gone with other colors, but I decided to go with orange.


## Summary of Work

This notebook demonstrates the process of fine-tuning an OpenAI language model for a telecom customer support use case. The main steps covered are:

1.  **Preparation of Training Data**: Customer complaint data from `telecom_fine_tuning.csv` was loaded and transformed into a JSONL format required by OpenAI's fine-tuning API. This involved structuring each training example with `system` (instructions for the model), `user` (customer complaint), and `assistant` (desired classification and response draft) roles.

2.  **Uploading Training Data and Fine-Tuning**: The prepared JSONL file was intended to be uploaded to OpenAI, and a fine-tuning job was initiated using a base model like `gpt-4o-mini`. (Note: The API key provided in the notebook resulted in authentication errors, preventing successful upload and fine-tuning in this run).

3.  **Using the Fine-Tuned Model**: A pre-existing fine-tuned model ID (`ft:gpt-4o-mini-2024-07-18:personal::DS51Qedq`) was used to generate responses to a sample customer complaint, demonstrating how the model can classify issues, assign severity, and draft empathetic responses.

4.  **Model Evaluation**: The performance of the custom fine-tuned model was compared against a standard `gpt-4o-mini` model using several NLP evaluation metrics:
    *   **ROUGE Score**: Measured the overlap of n-grams between the generated responses and a reference response. The custom model showed significantly higher ROUGE scores (e.g., rouge1: 0.81 vs. 0.34 for the standard model).
    *   **BLEU Score**: Assessed the precision of n-grams, penalizing brevity. The custom model achieved a much higher BLEU score (0.55 vs. 0.037).
    *   **METEOR Score**: Focused on semantic similarity using stemming, synonyms, and paraphrases. The custom model again outperformed the standard model (0.81 vs. 0.31).

Overall, the evaluation metrics suggest that the fine-tuned model produced responses that were much closer to the desired reference output compared to the standard model, highlighting the benefits of supervised fine-tuning for specific tasks.